<a href="https://colab.research.google.com/github/santiagodie/face-similarity-search-pipeline/blob/main/space/build_demo_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build the synthetic-face demo index

Builds the index for the Hugging Face Space `santiagodie/face-similarity-search-demo` from synthetic faces of [SFHQ part 3](https://www.kaggle.com/datasets/selfishgene/synthetic-faces-high-quality-sfhq-part-3) (CC0), using the pipeline from notebooks 02-04, and uploads it together with the contents of `space/`.

Before running: create the Space by hand (Gradio SDK) and add a Hugging Face write token as the Colab secret `HF_TOKEN`. Then use *Runtime → Run all*. Kaggle rate-limits anonymous downloads, so the index usually contains only part of the 750 faces.

In [ ]:
# 1. Install dependencies (same versions as space/requirements.txt) and get the repository
!pip uninstall -y -q dlib
!pip install -q numpy==2.2.6 opencv-python==4.12.0.88 tensorflow==2.21.0 tf-keras==2.21.0 deepface==0.0.100 dlib-bin==20.0.1 vptree==1.3 huggingface_hub==1.33.0
!git clone -q --depth 1 https://github.com/santiagodie/face-similarity-search-pipeline.git
%cd face-similarity-search-pipeline

In [ ]:
# 2. Download the SFHQ part 3 sample from Kaggle without authentication; failed files are skipped
import shutil
import urllib.parse
import urllib.request
from pathlib import Path

CARAS_DIR = Path("/content/sfhq")
CARAS_DIR.mkdir(exist_ok=True)
for ejemplo in Path("space/examples").glob("*.jpg"):   # the six example faces are already in the repo
    shutil.copy(ejemplo, CARAS_DIR / ejemplo.name)

nombres = [l.strip() for l in open("space/sfhq_pt3_sample_manifest.txt") if l.strip() and not l.startswith("#")]
fallos = 0
for nombre in nombres:
    if (CARAS_DIR / nombre).exists():
        continue
    ruta = urllib.parse.quote(f"a small samples (750 images)/{nombre}", safe="")
    url = f"https://www.kaggle.com/api/v1/datasets/download/selfishgene/synthetic-faces-high-quality-sfhq-part-3/{ruta}"
    try:
        datos = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})).read()
        (CARAS_DIR / nombre).write_bytes(datos)
    except Exception:
        fallos += 1
print(f"Faces available: {len(list(CARAS_DIR.glob('*.jpg')))} of {len(nombres)} ({fallos} downloads failed)")

In [ ]:
# 3. Variations, detection and 160x160 crops (code from notebooks 02 and 04)
import bz2
import cv2
import dlib
import numpy as np

urllib.request.urlretrieve("https://github.com/davisking/dlib-models/raw/master/mmod_human_face_detector.dat.bz2",
                           "mmod_human_face_detector.dat.bz2")
with bz2.open("mmod_human_face_detector.dat.bz2") as src, open("mmod_human_face_detector.dat", "wb") as dst:
    dst.write(src.read())
detectorHOG = dlib.get_frontal_face_detector()
detectorCNN = dlib.cnn_face_detection_model_v1("mmod_human_face_detector.dat")

def convert_and_trim_bb(image, rect):
    startX = max(0, rect.left())
    startY = max(0, rect.top())
    endX   = min(rect.right(),  image.shape[1])
    endY   = min(rect.bottom(), image.shape[0])
    return (startX, startY, endX - startX, endY - startY)

def detectarCaras(imagen_bgr):
    rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    rects_hog = detectorHOG(rgb, 1)
    if len(rects_hog) > 0:
        return [convert_and_trim_bb(imagen_bgr, r) for r in rects_hog], "HOG"
    results_cnn = detectorCNN(rgb, 1)
    return [convert_and_trim_bb(imagen_bgr, r.rect) for r in results_cnn], "CNN"

def recortarYNormalizar(imagen_bgr, x, y, w, h):
    ih, iw      = imagen_bgr.shape[:2]
    margen_px_w = int(w * 0.20)
    margen_px_h = int(h * 0.20)
    x1 = max(0,  x - margen_px_w)
    y1 = max(0,  y - margen_px_h)
    x2 = min(iw, x + w + margen_px_w)
    y2 = min(ih, y + h + margen_px_h)
    if (x2 - x1) < 20 or (y2 - y1) < 20:
        return None
    return cv2.resize(imagen_bgr[y1:y2, x1:x2], (160, 160),
                      interpolation=cv2.INTER_CUBIC)

def variaciones(img):
    h, w = img.shape[:2]
    m = int(0.08 * min(h, w))
    return {
        "orig":   img,
        "crop":   cv2.resize(img[m:h - m, m:w - m], (w, h)),
        "rot":    cv2.warpAffine(img, cv2.getRotationMatrix2D((w / 2, h / 2), 8, 1.0), (w, h),
                                 borderMode=cv2.BORDER_REFLECT),
        "bright": cv2.convertScaleAbs(img, alpha=1.3, beta=0),
        "flip":   cv2.flip(img, 1),
    }

RECORTES = Path("space/carasRecortadas")
RECORTES.mkdir(exist_ok=True)
for ruta in sorted(CARAS_DIR.glob("*.jpg")):
    for variacion, img in variaciones(cv2.imread(str(ruta))).items():
        h, w = img.shape[:2]
        if max(h, w) > 600:            # same downscaling as the Gradio app
            scale = 600 / max(h, w)
            img = cv2.resize(img, (int(w * scale), int(h * scale)))
        caras, _ = detectarCaras(img)
        if not caras:
            continue
        x, y, w_, h_ = max(caras, key=lambda c: c[2] * c[3])   # one synthetic face per image
        recorte = recortarYNormalizar(img, x, y, w_, h_)
        if recorte is not None:
            cv2.imwrite(str(RECORTES / f"{ruta.stem}__{variacion}.jpg"), recorte, [cv2.IMWRITE_JPEG_QUALITY, 95])
print(f"Crops: {len(list(RECORTES.glob('*.jpg')))}")

In [ ]:
# 4. FaceNet embeddings and VP-tree (notebooks 03 and 04)
import pickle
import vptree
from deepface import DeepFace

paths = sorted(p.name for p in RECORTES.glob("*.jpg"))
vecs_arr = np.array([DeepFace.represent(img_path=str(RECORTES / p), model_name="Facenet",
                                        enforce_detection=False)[0]["embedding"] for p in paths],
                    dtype=np.float32)
identidades = [p.split("__")[0] for p in paths]

def distancia(a, b):
    va = vecs_arr[int(a)] if isinstance(a, (int, np.integer)) else np.array(a, dtype=np.float32)
    vb = vecs_arr[int(b)] if isinstance(b, (int, np.integer)) else np.array(b, dtype=np.float32)
    return float(np.linalg.norm(va - vb))

tree = vptree.VPTree(list(range(len(paths))), distancia)
print("Top 5 for", paths[0], [(round(d, 2), paths[i]) for d, i in tree.get_n_nearest_neighbors(vecs_arr[0], 5)])

# The VP-tree is not picklable with this distance function (notebook 05), so the app
# rebuilds it at startup from vecs_arr; each embedding keeps its crop file and identity.
with open("space/vptree.pkl", "wb") as f:
    pickle.dump({"paths": paths, "vecs_arr": vecs_arr, "identities": identidades}, f)
print(f"Saved space/vptree.pkl: {vecs_arr.shape[0]} embeddings of {len(set(identidades))} faces")

In [ ]:
# 5. Upload space/ and the index to the Space (created beforehand by hand)
from google.colab import userdata
from huggingface_hub import HfApi

HfApi(token=userdata.get("HF_TOKEN")).upload_folder(
    folder_path="space",
    repo_id="santiagodie/face-similarity-search-demo",
    repo_type="space",
    ignore_patterns=["*.ipynb", "__pycache__/*"],
)
print("https://huggingface.co/spaces/santiagodie/face-similarity-search-demo")